<a href="https://colab.research.google.com/github/lricci03/challengedata/blob/main/CFM2024-new/cfm2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CFM 2024 Datachallenge

Models:
* Simple RNN, 1-layer GRU, 1-layer LSTM, simplified WaveNet
* 2-layers GRU (Deep GRU Model): trained for 60 epochs, validation accuracy 61%.

In [1]:
import torch
import torch.nn as nn

if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

In [2]:
import pandas as pd

In [3]:
from google.colab import drive

# drive.flush_and_unmount()
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [4]:
from pathlib import Path

DATA_DIR = Path(
    "/content/drive/MyDrive/Challengedata"
)

print(DATA_DIR.exists())

True


The original csv file is 1.2GB. Locally in CSV we convert it into a parquet file

```
import duckdb

# convert X_train_N1UvY30.csv into Xtrain.parquet
duckdb.sql("""
COPY (
    SELECT *
    FROM read_csv_auto('../X_train_N1UvY30.csv')
)
TO 'Xtrain.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")
```



Upload the data in a dataframe and look at the first 10 rows.

In [5]:
path = Path("/content/drive/MyDrive/Challengedata/Xtrain.parquet")

df_features = pd.read_parquet(path)

df_features.head()

,obs_id,venue,order_id,action,side,price,bid,ask,bid_size,ask_size,trade,flux
0,0,4,0,A,A,0.30,0.0,0.01,100,1,False,100
1,0,4,1,A,B,-0.17,0.0,0.01,100,1,False,100
2,0,4,2,D,A,0.28,0.0,0.01,100,1,False,-100
3,0,4,3,A,A,0.30,0.0,0.01,100,1,False,100
4,0,4,4,D,A,0.37,0.0,0.01,100,1,False,-100


In [6]:
df_features.shape

(16080000, 12)

In [7]:
df_features.columns

Index(['obs_id', 'venue', 'order_id', 'action', 'side', 'price', 'bid', 'ask',
       'bid_size', 'ask_size', 'trade', 'flux'],
      dtype='object')

Upload the targets

In [8]:
df_targets = pd.read_csv("/content/drive/MyDrive/Challengedata/y_train_or6m3Ta.csv")

df_targets.head()

,obs_id,eqt_code_cat
0,0,10
1,1,15
2,2,0
3,3,13
4,4,0


`order_id` are too many for one-hot encoding and most importantly it is preserved across observations.

We extract two informations out of it:
- order_seen_before
- order_previous_count

Since we want to create these features as well for the test set later on, we define a function that does it.

In [9]:
def add_order_features(df):

    # how many previous events involved this same order
    df["order_previous_count"] = (
        df.groupby(["obs_id", "order_id"])
          .cumcount()
          .astype("float32")
    )

    # whether this order was seen before
    df["order_seen_before"] = (
        df["order_previous_count"] > 0
    ).astype("float32")

    # raw ID no longer needed
    df.drop("order_id", axis=1, inplace=True)

    return df

In [10]:
df_features = add_order_features(df_features)

One-hot-encoding

In [11]:
df_features = df_features[[
    "obs_id",
    "price", "bid", "ask",
    "bid_size", "ask_size",
    "flux",
    "action", "venue", "side", "trade",
    "order_previous_count",
    "order_seen_before"
]]

In [12]:
num_attribs = ['price', 'bid', 'ask', 'bid_size', 'ask_size', 'flux', 'trade', 'order_seen_before', 'order_previous_count']
cat_attribs = ['venue', 'action', 'side']

In [13]:
df_features[num_attribs] = df_features[num_attribs].astype("float32")

In [14]:
df_features = pd.get_dummies(df_features, columns=cat_attribs, dtype="float32")

## Train and Validation Split

In [15]:
df_features = df_features.set_index("obs_id")
df_targets = df_targets.set_index("obs_id")

In [16]:
from sklearn.model_selection import train_test_split

train_ids, val_ids = train_test_split(
    df_targets.index,
    test_size=0.2,
    random_state=42
)

In [17]:
X_train = df_features.loc[train_ids]
X_val   = df_features.loc[val_ids]

y_train = df_targets.loc[train_ids]
y_val   = df_targets.loc[val_ids]

In [18]:
print(df_features.isna().sum())

price                   0
bid                     0
ask                     0
bid_size                0
ask_size                0
flux                    0
trade                   0
order_previous_count    0
order_seen_before       0
venue_0                 0
venue_1                 0
venue_2                 0
venue_3                 0
venue_4                 0
venue_5                 0
action_A                0
action_D                0
action_U                0
side_A                  0
side_B                  0
dtype: int64


## Convert to tensors

Number of features

In [19]:
n_features = X_train.shape[1]

print(n_features)

20


In [20]:
# Reshape X. This reinterprets every consecutive block of 100 rows as one sample.
# The *-1* means: infer this dimension automatically

X_train = torch.tensor(
    X_train.to_numpy(),
    dtype=torch.float32
).reshape(-1, 100, n_features)

X_val = torch.tensor(
    X_val.to_numpy(),
    dtype=torch.float32
).reshape(-1, 100, n_features)

# For classification, the targets should be integers, not floats -> `dtype=torch.long`

y_train = torch.tensor(
    y_train["eqt_code_cat"].to_numpy(),
    dtype=torch.long
)

y_val = torch.tensor(
    y_val["eqt_code_cat"].to_numpy(),
    dtype=torch.long
)

In [21]:
print(torch.isnan(X_train).any())
print(torch.isnan(X_val).any())

tensor(False)
tensor(False)


Delete the dataframes

In [22]:
import gc

del df_features, df_targets
gc.collect()

0

In [23]:
print("train_set shape:", X_train.shape)
print("y_train shape:", y_train.shape)

train_set shape: torch.Size([128640, 100, 20])
y_train shape: torch.Size([128640])


## Dataset and Dataloader

This is not needed if we already reshaped X. We can just use TensorDataset.

In [27]:
class StockDataset(torch.utils.data.Dataset):
  def __init__(self, features, targets):
    self.features = features
    self.targets = targets

  def __len__(self):
    return len(self.targets) # number of samples

  def __getitem__(self, idx):
    window = self.features[100* idx: 100*(idx + 1)]
    target = self.targets[idx]
    return window, target

In [24]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [25]:
valid_dataset = TensorDataset(X_val, y_val)
valid_loader = DataLoader(valid_dataset, batch_size=32)

Shape of dataset features and shape of dataset output

In [26]:
train_dataset[0][0].shape, train_dataset[0][1].shape

(torch.Size([100, 20]), torch.Size([]))

The shape of the target is a scalar, which is correct for `CrossEntropyLoss` when we do a classification problem.

## Train and evaluation functions

In [27]:
%pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 11.6 MB/s eta 0:00:00


In [28]:
import torchmetrics
import copy

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)

            y_class = y_pred.argmax(dim=1)
            metric.update(y_class, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=20, factor=0.1):

    best_valid = -float("inf")
    best_model_state = None
    best_epoch = None

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max", # we monitor accuracy
        patience=patience,
        factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)

            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            y_class = y_pred.detach().argmax(dim=1)
            metric.update(y_class, y_batch)

        train_loss = total_loss / len(train_loader)
        train_metric = metric.compute().item()
        val_metric = evaluate_tm(model, valid_loader, metric).item()

        if val_metric > best_valid:
          best_valid = val_metric
          best_model_state = copy.deepcopy(model.state_dict())
          best_epoch = epoch +1 # +1 bc epoch starts at 0

        history["train_losses"].append(train_loss)
        history["train_metrics"].append(train_metric)
        history["valid_metrics"].append(val_metric)

        scheduler.step(val_metric)


        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")

    model.load_state_dict(best_model_state)
    print("Best epoch:", best_epoch)
    print("Best validation accuracy:", best_valid)

    return history

## Comparing different models

### Simple RNN model

In [ ]:
class SimpleRnnModel(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.hidden_size = hidden_size
    self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
    self.output = nn.Linear(hidden_size, output_size)

  def forward(self,X):
    outputs, last_state = self.rnn(X)
    return self.output(outputs[:, -1])

input_size = number of features is 15

output_size = number of classes is 24

In [ ]:
torch.manual_seed(42)

simple_model = SimpleRnnModel(input_size=15, hidden_size=32, output_size=24).to(device)

In [ ]:
torch.manual_seed(42)

optimizer = torch.optim.NAdam(simple_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=24).to(device)

history = train(simple_model, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=50)

Epoch 1/50, train loss: 3.1841, train metric: 0.0515, valid metric: 0.0631
Epoch 2/50, train loss: 3.1056, train metric: 0.0674, valid metric: 0.0729
Epoch 3/50, train loss: 3.0011, train metric: 0.0783, valid metric: 0.0849
Epoch 4/50, train loss: 2.9319, train metric: 0.0871, valid metric: 0.0908
Epoch 5/50, train loss: 2.9084, train metric: 0.0915, valid metric: 0.0914
Epoch 6/50, train loss: 2.8959, train metric: 0.0929, valid metric: 0.0938
Epoch 7/50, train loss: 2.8891, train metric: 0.0947, valid metric: 0.0930
Epoch 8/50, train loss: 2.8839, train metric: 0.0949, valid metric: 0.0975
Epoch 9/50, train loss: 2.8796, train metric: 0.0961, valid metric: 0.0961
Epoch 10/50, train loss: 2.8757, train metric: 0.0978, valid metric: 0.0966
Epoch 11/50, train loss: 2.8723, train metric: 0.0980, valid metric: 0.0987
Epoch 12/50, train loss: 2.8683, train metric: 0.0990, valid metric: 0.0999
Epoch 13/50, train loss: 2.8632, train metric: 0.1001, valid metric: 0.1001
Epoch 14/50, train lo

Look at some predictions

In [ ]:
for idx in range(10):
    X, y = valid_dataset[idx]
    X = X.to(device)
    y = y.to(device)

    with torch.no_grad():
        logits = simple_model(X.unsqueeze(0)) # add batch dimension: (100,15) -> (1,100,15)
        pred = logits.argmax(dim=1).item()

    print(f"idx={idx}, predicted={pred}, actual={y.item()}")

idx=0, predicted=15, actual=2
idx=1, predicted=18, actual=18
idx=2, predicted=3, actual=8
idx=3, predicted=21, actual=1
idx=4, predicted=21, actual=6
idx=5, predicted=5, actual=18
idx=6, predicted=3, actual=4
idx=7, predicted=5, actual=7
idx=8, predicted=13, actual=13
idx=9, predicted=23, actual=1


### GRU MODEL

In [ ]:
class GRUModel(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.hidden_size = hidden_size
    self.rnn = nn.GRU(input_size, hidden_size, batch_first=True)
    self.output = nn.Linear(hidden_size, output_size)

  def forward(self,X):
    outputs, last_state = self.rnn(X)
    return self.output(outputs[:, -1])

In [ ]:
torch.manual_seed(42)

gru_model = GRUModel(input_size=15, hidden_size=32, output_size=24).to(device)

In [ ]:
torch.manual_seed(42)

optimizer = torch.optim.NAdam(gru_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=24).to(device)

gru_history = train(gru_model, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=50)

Epoch 1/50, train loss: 3.0683, train metric: 0.0698, valid metric: 0.0836
Epoch 2/50, train loss: 2.9310, train metric: 0.0889, valid metric: 0.0973
Epoch 3/50, train loss: 2.8508, train metric: 0.1025, valid metric: 0.1126
Epoch 4/50, train loss: 2.7615, train metric: 0.1200, valid metric: 0.1230
Epoch 5/50, train loss: 2.6931, train metric: 0.1296, valid metric: 0.1311
Epoch 6/50, train loss: 2.6494, train metric: 0.1363, valid metric: 0.1410
Epoch 7/50, train loss: 2.6174, train metric: 0.1433, valid metric: 0.1466
Epoch 8/50, train loss: 2.5890, train metric: 0.1495, valid metric: 0.1499
Epoch 9/50, train loss: 2.5642, train metric: 0.1549, valid metric: 0.1578
Epoch 10/50, train loss: 2.5421, train metric: 0.1590, valid metric: 0.1639
Epoch 11/50, train loss: 2.5212, train metric: 0.1653, valid metric: 0.1660
Epoch 12/50, train loss: 2.5022, train metric: 0.1696, valid metric: 0.1709
Epoch 13/50, train loss: 2.4842, train metric: 0.1759, valid metric: 0.1774
Epoch 14/50, train lo

In [ ]:
for idx in range(10):
    X, y = valid_dataset[idx]
    X = X.to(device)
    y = y.to(device)

    with torch.no_grad():
        gru_logits = gru_model(X.unsqueeze(0)) # add batch dimension: (100,15) -> (1,100,15)
        rnn_logits = simple_model(X.unsqueeze(0))
        gru_pred = gru_logits.argmax(dim=1).item()
        rnn_pred = rnn_logits.argmax(dim=1).item()

    print(f"idx={idx}, gru predicted={gru_pred}, rnn predicted={rnn_pred} actual={y.item()}")

idx=0, gru predicted=2, rnn predicted=15 actual=2
idx=1, gru predicted=10, rnn predicted=18 actual=18
idx=2, gru predicted=8, rnn predicted=3 actual=8
idx=3, gru predicted=1, rnn predicted=21 actual=1
idx=4, gru predicted=19, rnn predicted=21 actual=6
idx=5, gru predicted=4, rnn predicted=5 actual=18
idx=6, gru predicted=15, rnn predicted=3 actual=4
idx=7, gru predicted=2, rnn predicted=5 actual=7
idx=8, gru predicted=13, rnn predicted=13 actual=13
idx=9, gru predicted=23, rnn predicted=23 actual=1


### LSTM Model

In [ ]:
class LSTMModel(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.hidden_size = hidden_size
    self.rnn = nn.LSTM(input_size, hidden_size, batch_first=True)
    self.output = nn.Linear(hidden_size, output_size)

  def forward(self,X):
    outputs, last_state = self.rnn(X)
    return self.output(outputs[:, -1])

In [ ]:
torch.manual_seed(42)

lstm_model = LSTMModel(input_size=15, hidden_size=32, output_size=24).to(device)

In [ ]:
torch.manual_seed(42)

optimizer = torch.optim.NAdam(lstm_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=24).to(device)

lstm_history = train(lstm_model, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=50)

Epoch 1/50, train loss: 3.1331, train metric: 0.0585, valid metric: 0.0751
Epoch 2/50, train loss: 2.9703, train metric: 0.0834, valid metric: 0.0909
Epoch 3/50, train loss: 2.8728, train metric: 0.0986, valid metric: 0.1065
Epoch 4/50, train loss: 2.8163, train metric: 0.1075, valid metric: 0.1102
Epoch 5/50, train loss: 2.8597, train metric: 0.1016, valid metric: 0.1123
Epoch 6/50, train loss: 2.7734, train metric: 0.1128, valid metric: 0.1191
Epoch 7/50, train loss: 2.7494, train metric: 0.1167, valid metric: 0.1212
Epoch 8/50, train loss: 2.7268, train metric: 0.1213, valid metric: 0.1176
Epoch 9/50, train loss: 2.7026, train metric: 0.1260, valid metric: 0.1244
Epoch 10/50, train loss: 2.6832, train metric: 0.1279, valid metric: 0.1268
Epoch 11/50, train loss: 2.6666, train metric: 0.1309, valid metric: 0.1293
Epoch 12/50, train loss: 2.6529, train metric: 0.1328, valid metric: 0.1288
Epoch 13/50, train loss: 2.6360, train metric: 0.1372, valid metric: 0.1333
Epoch 14/50, train lo

### CNN - simplified WaveNet
*WaveNet*: stacking 1D convolutional layers, doubling the dilation rate (how spread apart each neuron's inputs are) at every layer. The first conv layer sees two time steps at a time, the nex one sees four time steps, etc.

The lower layers learn short-term patterns, while the higher layers learn long-term patterns.

Custom CausalConv1d module that acts just like an nn.Conv1d module, except the inuts get padded on the left side by the appropriate amount to ensure the sequence preserves the same length.

- kernel_size will be 2 for each layer
- dilation will increase: 1,2,4,8 and then again 1,2,4,8

In [ ]:
import torch.nn.functional as F

class CausalConv1d(nn.Conv1d):
  def forward(self, X):
    padding = (self.kernel_size[0] - 1) * self.dilation[0] # calculate the size of the left-padding we need
    X = F.pad(X, (padding, 0))  # X is the tensor to pad, (padding, 0) is how much to pad to the left and right in the time dimension
    return super().forward(X)

WaveNet model.

Must swap the two dimensions of the nn.Conv1d layer's inputs and outputs: nn.Conv1d expects inputs of shape [*batch size, input features, sequence length*] and produces outputs of shape [*batch size, output features, sequence length*]

while we have inputs [*batch size, sequence length, input features*] (with sequence length 100, input features 15).

We don't need the prediction for all 100 time steps, so we then take the last time step: Z[:, :, -1].

In [ ]:
class WavenetModel(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    layers = []
    for dilation in (1,2,4,8) * 2: # (1,2,4,8,1,2,4,8)
      conv = CausalConv1d(input_size, hidden_size, kernel_size=2, dilation=dilation)
      layers += [conv, nn.ReLU()]
      input_size = hidden_size
    self.convs = nn.Sequential(*layers)
    self.output = nn.Linear(hidden_size, output_size)

  def forward(self,X):
    Z = X.permute(0,2,1)
    Z = self.convs(Z)
    Z = Z[:, :, -1]
    return self.output(Z)

In [ ]:
torch.manual_seed(42)
wavenet_model = WavenetModel(input_size=15, hidden_size=32, output_size=24).to(device)

optimizer = torch.optim.NAdam(wavenet_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=24).to(device)

wavenet_history = train(wavenet_model, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=50)

Epoch 1/50, train loss: 2.9142, train metric: 0.0848, valid metric: 0.1032
Epoch 2/50, train loss: 2.7814, train metric: 0.1061, valid metric: 0.1126
Epoch 3/50, train loss: 2.7440, train metric: 0.1164, valid metric: 0.1087
Epoch 4/50, train loss: 2.7223, train metric: 0.1219, valid metric: 0.1248
Epoch 5/50, train loss: 2.7039, train metric: 0.1277, valid metric: 0.1318
Epoch 6/50, train loss: 2.6913, train metric: 0.1303, valid metric: 0.1288
Epoch 7/50, train loss: 2.6810, train metric: 0.1330, valid metric: 0.1341
Epoch 8/50, train loss: 2.6719, train metric: 0.1353, valid metric: 0.1375
Epoch 9/50, train loss: 2.6642, train metric: 0.1371, valid metric: 0.1387
Epoch 10/50, train loss: 2.6568, train metric: 0.1403, valid metric: 0.1314
Epoch 11/50, train loss: 2.6495, train metric: 0.1417, valid metric: 0.1409
Epoch 12/50, train loss: 2.6431, train metric: 0.1436, valid metric: 0.1442
Epoch 13/50, train loss: 2.6340, train metric: 0.1471, valid metric: 0.1421
Epoch 14/50, train lo

## Comparing hidden sizes in GRU Model

Gru Model with hidden size = 64

In [ ]:
torch.manual_seed(42)

gru_model_64 = GRUModel(input_size=15, hidden_size=64, output_size=24).to(device)

optimizer = torch.optim.NAdam(gru_model_64.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=24).to(device)

gru_64_history = train(gru_model_64, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=50)

Epoch 1/50, train loss: 3.0117, train metric: 0.0807, valid metric: 0.1050
Epoch 2/50, train loss: 2.7969, train metric: 0.1153, valid metric: 0.1287
Epoch 3/50, train loss: 2.6821, train metric: 0.1352, valid metric: 0.1394
Epoch 4/50, train loss: 2.6198, train metric: 0.1474, valid metric: 0.1522
Epoch 5/50, train loss: 2.5692, train metric: 0.1583, valid metric: 0.1586
Epoch 6/50, train loss: 2.5222, train metric: 0.1698, valid metric: 0.1699
Epoch 7/50, train loss: 2.4754, train metric: 0.1812, valid metric: 0.1860
Epoch 8/50, train loss: 2.4354, train metric: 0.1936, valid metric: 0.1950
Epoch 9/50, train loss: 2.3985, train metric: 0.2021, valid metric: 0.1990
Epoch 10/50, train loss: 2.3649, train metric: 0.2135, valid metric: 0.2133
Epoch 11/50, train loss: 2.3332, train metric: 0.2212, valid metric: 0.2263
Epoch 12/50, train loss: 2.3029, train metric: 0.2336, valid metric: 0.2262
Epoch 13/50, train loss: 2.2760, train metric: 0.2411, valid metric: 0.2450
Epoch 14/50, train lo

Gru model with hidden size 128

In [ ]:
torch.manual_seed(42)

gru_model_128 = GRUModel(input_size=15, hidden_size=128, output_size=24).to(device)

optimizer = torch.optim.NAdam(gru_model_128.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=24).to(device)

gru_128_history = train(gru_model_128, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=50)

Epoch 1/50, train loss: 2.9319, train metric: 0.0948, valid metric: 0.1212
Epoch 2/50, train loss: 2.6910, train metric: 0.1376, valid metric: 0.1503
Epoch 3/50, train loss: 2.5801, train metric: 0.1617, valid metric: 0.1625
Epoch 4/50, train loss: 2.5119, train metric: 0.1798, valid metric: 0.1840
Epoch 5/50, train loss: 2.4495, train metric: 0.1967, valid metric: 0.1990
Epoch 6/50, train loss: 2.3915, train metric: 0.2133, valid metric: 0.2131
Epoch 7/50, train loss: 2.3347, train metric: 0.2302, valid metric: 0.2362
Epoch 8/50, train loss: 2.2794, train metric: 0.2499, valid metric: 0.2586
Epoch 9/50, train loss: 2.2235, train metric: 0.2676, valid metric: 0.2685
Epoch 10/50, train loss: 2.1689, train metric: 0.2842, valid metric: 0.2921
Epoch 11/50, train loss: 2.1205, train metric: 0.2995, valid metric: 0.2984
Epoch 12/50, train loss: 2.0797, train metric: 0.3108, valid metric: 0.3157
Epoch 13/50, train loss: 2.0435, train metric: 0.3213, valid metric: 0.3237
Epoch 14/50, train lo

#### Save the model weights

In [ ]:
torch.save(
    gru_model_128.state_dict(),
    "/content/drive/MyDrive/Challengedata/gru_128_model.pt"
)

#### Upload the model weights

In [ ]:
gru_model_128 = GRUModel(
    input_size=15,
    hidden_size=128,
    output_size=24
)

gru_model_128.load_state_dict(
    torch.load("/content/drive/MyDrive/Challengedata/gru_128_model.pt", map_location=device)
)

gru_model_128.to(device)
gru_model_128.eval()

## Create submission

We first need to prepreocess the test data.

We predict the test set using `gru_model_128` and create the submission file.

In [ ]:
test = pd.read_parquet("/content/drive/MyDrive/Challengedata/Xtest.parquet")

In [ ]:
test = test[['obs_id','price', 'bid', 'ask', 'bid_size', 'ask_size', 'flux', 'action', 'venue']]
test['venue'] = test['venue'].apply(str)
test = pd.get_dummies(test, dtype=float)
test = test.set_index('obs_id')

In [ ]:
X_test = torch.tensor(test.values, dtype=torch.float32)
X_test = X_test.reshape(-1, 100, 15)

In [ ]:
del test
gc.collect()

266

Create a test_loader because we want to predict in batches rather than send all the data to the GPU at once.

In [ ]:
test_dataset = TensorDataset(X_test)
test_loader = DataLoader(test_dataset, batch_size=256)

Predict the data

In [ ]:
gru_model_128.eval()

predictions = []

with torch.no_grad():
    for (X_batch,) in test_loader:
        X_batch = X_batch.to(device)

        logits = gru_model_128(X_batch)
        y_pred = logits.argmax(dim=1)

        predictions.extend(y_pred.cpu().numpy())

Save the predictions to csv

In [ ]:
submission = pd.DataFrame({
    "obs_id": range(len(predictions)),
    "eqt_code_cat": predictions
})

submission.to_csv(
    "/content/drive/MyDrive/Challengedata/submission.csv",
    index=False
)

## Deep GRU Model

In [29]:
class DeepGRUModel(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.hidden_size = hidden_size
    self.rnn = nn.GRU(input_size, hidden_size, num_layers=2, batch_first=True)
    self.output = nn.Linear(hidden_size, output_size)

  def forward(self,X):
    outputs, last_state = self.rnn(X)
    return self.output(outputs[:, -1])

In [30]:
torch.manual_seed(42)

deep_gru_model_128 = DeepGRUModel(input_size=20, hidden_size=128, output_size=24).to(device)

optimizer = torch.optim.NAdam(deep_gru_model_128.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=24).to(device)

history = train(deep_gru_model_128, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=50)

Epoch 1/50, train loss: 2.7282, train metric: 0.1251, valid metric: 0.1545
Epoch 2/50, train loss: 2.5432, train metric: 0.1694, valid metric: 0.1809
Epoch 3/50, train loss: 2.4503, train metric: 0.1951, valid metric: 0.1991
Epoch 4/50, train loss: 2.3820, train metric: 0.2115, valid metric: 0.2189
Epoch 5/50, train loss: 2.3161, train metric: 0.2323, valid metric: 0.2440
Epoch 6/50, train loss: 2.2294, train metric: 0.2605, valid metric: 0.2656
Epoch 7/50, train loss: 2.1502, train metric: 0.2816, valid metric: 0.2865
Epoch 8/50, train loss: 2.0827, train metric: 0.3021, valid metric: 0.2975
Epoch 9/50, train loss: 2.0187, train metric: 0.3207, valid metric: 0.3181
Epoch 10/50, train loss: 1.9554, train metric: 0.3428, valid metric: 0.3466
Epoch 11/50, train loss: 1.8979, train metric: 0.3620, valid metric: 0.3584
Epoch 12/50, train loss: 1.8422, train metric: 0.3809, valid metric: 0.3771
Epoch 13/50, train loss: 1.7906, train metric: 0.3989, valid metric: 0.4001
Epoch 14/50, train lo

In [31]:
history_2 = train(deep_gru_model_128, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=10)

Epoch 1/10, train loss: 1.0060, train metric: 0.6559, valid metric: 0.5896
Epoch 2/10, train loss: 0.9963, train metric: 0.6588, valid metric: 0.6036
Epoch 3/10, train loss: 0.9874, train metric: 0.6608, valid metric: 0.6084
Epoch 4/10, train loss: 0.9812, train metric: 0.6635, valid metric: 0.6039
Epoch 5/10, train loss: 0.9696, train metric: 0.6663, valid metric: 0.6000
Epoch 6/10, train loss: 0.9614, train metric: 0.6697, valid metric: 0.6113
Epoch 7/10, train loss: 0.9535, train metric: 0.6722, valid metric: 0.6105
Epoch 8/10, train loss: 0.9441, train metric: 0.6746, valid metric: 0.6053
Epoch 9/10, train loss: 0.9372, train metric: 0.6770, valid metric: 0.6135
Epoch 10/10, train loss: 0.9285, train metric: 0.6808, valid metric: 0.6097
Best epoch: 9
Best validation accuracy: 0.6135261058807373


### Save & load model

Save the model: this is trained **without normalization of the inputs** and with 2 gru layers.

In [32]:
torch.save(
    deep_gru_model_128.state_dict(),
    "/content/drive/MyDrive/Challengedata/deep_gru_model_128.pt"
)

Load the model

In [ ]:
deep_gru_model_128.load_state_dict(
    torch.load("/content/drive/MyDrive/Challengedata/deep_gru_model_128.pt")
)